# Refresh / Content Opportunity Scoring — Capstone Pipeline

## Data Ingestion & Safety Guidelines

> ⚠️ **Data Safety Guardrail:** As per FlyRank safety guidelines and the data contract, we strictly avoid using target-leaking features such as `trend_direction` and `trend_pct` in feature engineering and model training. The target label (`is_declining_label` / `is_decaying`) is mathematically derived from performance trends, so including them would cause direct target leakage. Furthermore, pseudonymous IDs (`content_id`, `client_id`) are reserved exclusively for grouping, partitioning, and joins—never as predictive features.

In [1]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

assert HF_TOKEN, "HF_TOKEN not found — set it as an environment variable (.env) or Colab secret."

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

WH = "hf://datasets/FlyRank/internship-warehouse"
FACT_PERF = f"{WH}/fact_content_daily_performance"

sample_query = f"""
    SELECT *
    FROM '{FACT_PERF}/month=2026-03/*.parquet'
    LIMIT 5
"""

df_sample = con.execute(sample_query).fetchdf()

print("--- Dataset Schema ---")
print(df_sample.dtypes)
print("\n--- First 5 Rows ---")
if 'display' in globals():
    display(df_sample)
else:
    print(df_sample)


--- Dataset Schema ---
report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                 boolean
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     int64
gsc_avg_position                   float64
ga4_pageviews                        Int64
ga4_sessions                         Int64
ga4_users                            Int64
ga4_engaged_sessions                 Int64
ga4_total_engagement_sec             Int64
sessions_organic                     Int64
sessions_direct                      Int64
sessions_referral                    Int64
sessions_social                      Int64
sessions_paid                        Int64
sessions_ai                          Int64
ai_chatgpt                     

## Target Label Definition

**Target Label (`is_decaying`):** 1 if `gsc_clicks` dropped by >= 20% between the previous 14 days and the last 14 days (`click_change_pct <= -0.20`), otherwise 0.

- **`current_window`**: The most recent 14 days of the performance dataset (`(max_date - 14 days, max_date]`).
- **`previous_window`**: The 14 days before that (`(max_date - 28 days, max_date - 14 days]`).
- **Threshold Decision**: A 20% drop threshold captures meaningful business risk and decay while providing a well-balanced distribution for modeling.
- **Noise / Sparsity Filter**: Excludes items with `previous_clicks < 10` to avoid division by zero and noisy percentage calculations.

In [2]:
target_label_query = f"""
WITH date_bounds AS (
    SELECT MAX(report_date) AS max_date
    FROM '{FACT_PERF}/month=2026-03/*.parquet'
),
daily_windowed AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_clicks,
        gsc_impressions,
        CASE
            WHEN report_date > (SELECT max_date - INTERVAL '14 days' FROM date_bounds)
                 AND report_date <= (SELECT max_date FROM date_bounds)
                THEN 'current_window'
            WHEN report_date > (SELECT max_date - INTERVAL '28 days' FROM date_bounds)
                 AND report_date <= (SELECT max_date - INTERVAL '14 days' FROM date_bounds)
                THEN 'previous_window'
            ELSE NULL
        END AS window_bucket
    FROM '{FACT_PERF}/month=2026-03/*.parquet'
),
content_aggregated AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN gsc_clicks ELSE 0 END) AS previous_clicks,
        SUM(CASE WHEN window_bucket = 'current_window' THEN gsc_clicks ELSE 0 END) AS current_clicks,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN gsc_impressions ELSE 0 END) AS previous_impressions,
        SUM(CASE WHEN window_bucket = 'current_window' THEN gsc_impressions ELSE 0 END) AS current_impressions
    FROM daily_windowed
    WHERE window_bucket IS NOT NULL
    GROUP BY content_hash_id
)
SELECT
    content_hash_id,
    client_hash_id,
    previous_clicks,
    current_clicks,
    previous_impressions,
    current_impressions,
    ROUND(((current_clicks - previous_clicks)::DOUBLE / previous_clicks), 4) AS click_change_pct,
    CASE 
        WHEN ((current_clicks - previous_clicks)::DOUBLE / previous_clicks) <= -0.20 THEN 1 
        ELSE 0 
    END AS is_decaying
FROM content_aggregated
WHERE previous_clicks >= 10
ORDER BY previous_clicks DESC
"""

df_labeled = con.execute(target_label_query).fetchdf()

print(f"Total labeled content items: {len(df_labeled):,}")
print("\nDecay class distribution (Threshold <= -0.20):")
print(df_labeled['is_decaying'].value_counts())
print(df_labeled['is_decaying'].value_counts(normalize=True).round(4) * 100)
print("\nFirst 5 rows of df_labeled:")
if 'display' in globals():
    display(df_labeled.head(5))
else:
    print(df_labeled.head(5))


Total labeled content items: 8,844

Decay class distribution (Threshold <= -0.20):
is_decaying
0    4559
1    4285
Name: count, dtype: int64
is_decaying
0    51.55
1    48.45
Name: proportion, dtype: float64

First 5 rows of df_labeled:
            content_hash_id           client_hash_id  previous_clicks  \
0  content_eadb33b5df496f4a  client_e547b89c05043229           2450.0   
1  content_512dbad65bd5ade9  client_73cda7b4e4f265ea           1012.0   
2  content_e7b5dd4dff461ad2  client_08a6a72ff48e62c0            942.0   
3  content_0ec90963d98b97a5  client_20259bd6705d81d4            777.0   
4  content_ec2e0346994fb5a5  client_e547b89c05043229            685.0   

   current_clicks  previous_impressions  current_impressions  \
0          2944.0              165685.0             430887.0   
1          1196.0               53634.0              80974.0   
2          1295.0               70358.0             105702.0   
3           487.0               60373.0              54088.0   
4   

## Feature Engineering (Leakage-Free Design)

To prevent target leakage, all predictive features are engineered strictly from the historical 14-day observation window (`previous_window`). 
The target label (`is_decaying`) is evaluated exclusively on the subsequent 14-day outcome window (`current_window`).

### Features Computed (Observation Window Only):
1. **`prev_clicks`**: Total Google Search Console clicks in the previous 14 days.
2. **`prev_impressions`**: Total GSC impressions in the previous 14 days.
3. **`prev_ctr`**: Click-through rate (`prev_clicks / prev_impressions`).
4. **`prev_avg_position`**: Impression-weighted average ranking position (`prev_sum_pos / prev_impressions`).
5. **`prev_engagement_rate`**: GA4 engaged sessions ratio (`prev_ga4_engaged_sessions / prev_ga4_sessions`).
6. **`prev_organic_ratio`**: Ratio of organic search sessions to total sessions (`prev_sessions_organic / prev_ga4_sessions`).

*Note: Missing GA4 engagement metrics (for content/clients without GA4 tracking) are imputed with 0.0.*

In [3]:
feature_engineering_query = f"""
WITH date_bounds AS (
    SELECT MAX(report_date) AS max_date
    FROM '{FACT_PERF}/month=2026-03/*.parquet'
),
daily_windowed AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_sum_position,
        ga4_sessions,
        ga4_engaged_sessions,
        sessions_organic,
        CASE
            WHEN report_date > (SELECT max_date - INTERVAL '14 days' FROM date_bounds)
                 AND report_date <= (SELECT max_date FROM date_bounds)
                THEN 'current_window'
            WHEN report_date > (SELECT max_date - INTERVAL '28 days' FROM date_bounds)
                 AND report_date <= (SELECT max_date - INTERVAL '14 days' FROM date_bounds)
                THEN 'previous_window'
            ELSE NULL
        END AS window_bucket
    FROM '{FACT_PERF}/month=2026-03/*.parquet'
),
content_aggregated AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN gsc_clicks ELSE 0 END) AS prev_clicks,
        SUM(CASE WHEN window_bucket = 'current_window' THEN gsc_clicks ELSE 0 END) AS curr_clicks,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN gsc_impressions ELSE 0 END) AS prev_impressions,
        SUM(CASE WHEN window_bucket = 'current_window' THEN gsc_impressions ELSE 0 END) AS curr_impressions,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN gsc_sum_position ELSE 0 END) AS prev_sum_pos,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN ga4_sessions ELSE 0 END) AS prev_ga4_sessions,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN ga4_engaged_sessions ELSE 0 END) AS prev_ga4_engaged_sessions,
        SUM(CASE WHEN window_bucket = 'previous_window' THEN sessions_organic ELSE 0 END) AS prev_sessions_organic
    FROM daily_windowed
    WHERE window_bucket IS NOT NULL
    GROUP BY content_hash_id
)
SELECT
    content_hash_id,
    client_hash_id,
    prev_clicks,
    prev_impressions,
    ROUND((prev_clicks::DOUBLE / NULLIF(prev_impressions, 0)), 4) AS prev_ctr,
    ROUND((prev_sum_pos::DOUBLE / NULLIF(prev_impressions, 0)), 2) AS prev_avg_position,
    ROUND((prev_ga4_engaged_sessions::DOUBLE / NULLIF(prev_ga4_sessions, 0)), 4) AS prev_engagement_rate,
    ROUND((prev_sessions_organic::DOUBLE / NULLIF(prev_ga4_sessions, 0)), 4) AS prev_organic_ratio,
    curr_clicks,
    ROUND(((curr_clicks - prev_clicks)::DOUBLE / prev_clicks), 4) AS click_change_pct,
    CASE 
        WHEN ((curr_clicks - prev_clicks)::DOUBLE / prev_clicks) <= -0.20 THEN 1 
        ELSE 0 
    END AS is_decaying
FROM content_aggregated
WHERE prev_clicks >= 10
ORDER BY prev_clicks DESC
"""

df_model = con.execute(feature_engineering_query).fetchdf()

df_model['prev_ctr'] = df_model['prev_ctr'].fillna(0.0)
df_model['prev_avg_position'] = df_model['prev_avg_position'].fillna(0.0)
df_model['prev_engagement_rate'] = df_model['prev_engagement_rate'].fillna(0.0)
df_model['prev_organic_ratio'] = df_model['prev_organic_ratio'].fillna(0.0)

print(f"Dataset shape: {df_model.shape}")
print(f"Missing values remaining:\n{df_model.isnull().sum()}")
print("\nFirst 5 rows of df_model:")
if 'display' in globals():
    display(df_model.head(5))
else:
    print(df_model.head(5))


Dataset shape: (8844, 11)
Missing values remaining:
content_hash_id         0
client_hash_id          0
prev_clicks             0
prev_impressions        0
prev_ctr                0
prev_avg_position       0
prev_engagement_rate    0
prev_organic_ratio      0
curr_clicks             0
click_change_pct        0
is_decaying             0
dtype: int64

First 5 rows of df_model:
            content_hash_id           client_hash_id  prev_clicks  \
0  content_eadb33b5df496f4a  client_e547b89c05043229       2450.0   
1  content_512dbad65bd5ade9  client_73cda7b4e4f265ea       1012.0   
2  content_e7b5dd4dff461ad2  client_08a6a72ff48e62c0        942.0   
3  content_0ec90963d98b97a5  client_20259bd6705d81d4        777.0   
4  content_ec2e0346994fb5a5  client_e547b89c05043229        685.0   

   prev_impressions  prev_ctr  prev_avg_position  prev_engagement_rate  \
0          165685.0    0.0148               2.39                0.0906   
1           53634.0    0.0189               3.01           

## Baseline Rule & Evaluation

To establish a transparent benchmark that ML models must beat, we define a heuristic rule baseline:
- **Heuristic Rule:** A page is predicted to decay (`is_decaying = 1`) if it has poor click-through performance and weak search ranking:
  $$\text{prev\_ctr} < 0.01 \quad \text{AND} \quad \text{prev\_avg\_position} > 10.0$$
- **Evaluation Split:** 80/20 Train/Test split (`stratify=y`, `random_state=42`).
- **Leakage Prevention:** Leaky columns (`curr_clicks`, `click_change_pct`) and identifier strings (`content_hash_id`, `client_hash_id`) are excluded from feature matrices.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

feature_cols = [
    'prev_clicks',
    'prev_impressions',
    'prev_ctr',
    'prev_avg_position',
    'prev_engagement_rate',
    'prev_organic_ratio'
]
target_col = 'is_decaying'

X = df_model[feature_cols]
y = df_model[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

y_pred_baseline = ((X_test['prev_ctr'] < 0.01) & (X_test['prev_avg_position'] > 10.0)).astype(int)

acc_base = accuracy_score(y_test, y_pred_baseline)
prec_base = precision_score(y_test, y_pred_baseline, zero_division=0)
rec_base = recall_score(y_test, y_pred_baseline, zero_division=0)
f1_base = f1_score(y_test, y_pred_baseline, zero_division=0)

print(f"Test Split Size: {len(y_test):,} samples")
print(f"Test Base Rate (Decaying %): {y_test.mean() * 100:.2f}%")
print("\n" + "="*45)
print("  BASELINE RULE PERFORMANCE (80/20 Test Split)")
print("="*45)
print(f"Accuracy:  {acc_base:.4f} ({acc_base*100:.2f}%)")
print(f"Precision: {prec_base:.4f} ({prec_base*100:.2f}%)")
print(f"Recall:    {rec_base:.4f} ({rec_base*100:.2f}%)")
print(f"F1-Score:  {f1_base:.4f}")
print("="*45)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=['Stable/Growing (0)', 'Decaying (1)'], digits=4))


Test Split Size: 1,769 samples
Test Base Rate (Decaying %): 48.45%

  BASELINE RULE PERFORMANCE (80/20 Test Split)
Accuracy:  0.5274 (52.74%)
Precision: 0.5303 (53.03%)
Recall:    0.2147 (21.47%)
F1-Score:  0.3056

Detailed Classification Report:
                    precision    recall  f1-score   support

Stable/Growing (0)     0.5267    0.8213    0.6418       912
      Decaying (1)     0.5303    0.2147    0.3056       857

          accuracy                         0.5274      1769
         macro avg     0.5285    0.5180    0.4737      1769
      weighted avg     0.5284    0.5274    0.4790      1769



## 4. Model & Analysis — Gradient Boosted Trees (XGBoost)

We train a non-linear gradient-boosted decision tree classifier (**XGBoost**) on our leakage-free engineered features (`prev_clicks`, `prev_impressions`, `prev_ctr`, `prev_avg_position`, `prev_engagement_rate`, `prev_organic_ratio`) to predict content decay (`is_decaying`, $\ge 20\%$ drop).

- **Objective:** Learn non-linear interaction effects between ranking position slippage, engagement loss, and volume to significantly outperform the heuristic baseline.
- **Evaluation Split:** Evaluated on the exact same 80/20 test split (`X_test`, `y_test`) as the baseline for honest, direct comparison.

In [5]:
!pip install xgboost -q

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

acc_xgb = accuracy_score(y_test, xgb_preds)
prec_xgb = precision_score(y_test, xgb_preds, zero_division=0)
rec_xgb = recall_score(y_test, xgb_preds, zero_division=0)
f1_xgb = f1_score(y_test, xgb_preds, zero_division=0)

print("="*50)
print("  XGBOOST MODEL PERFORMANCE (80/20 Test Split)")
print("="*50)
print(f"Accuracy:  {acc_xgb:.4f} ({acc_xgb*100:.2f}%)")
print(f"Precision: {prec_xgb:.4f} ({prec_xgb*100:.2f}%)")
print(f"Recall:    {rec_xgb:.4f} ({rec_xgb*100:.2f}%)")
print(f"F1-Score:  {f1_xgb:.4f}")
print("="*50)

lift_f1 = f1_xgb - f1_base
print(f"\n📊 Comparison vs Baseline:")
print(f"   • Baseline Rule F1-Score: {f1_base:.4f}")
print(f"   • XGBoost Model F1-Score: {f1_xgb:.4f}")
print(f"   • Net Lift in F1-Score:   {lift_f1:+.4f} ({(lift_f1 / f1_base) * 100:+.1f}% improvement)")

print("\nDetailed Classification Report:")
print(classification_report(y_test, xgb_preds, target_names=['Stable/Growing (0)', 'Decaying (1)'], digits=4))



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


  XGBOOST MODEL PERFORMANCE (80/20 Test Split)
Accuracy:  0.5257 (52.57%)
Precision: 0.5108 (51.08%)
Recall:    0.4959 (49.59%)
F1-Score:  0.5033

📊 Comparison vs Baseline:
   • Baseline Rule F1-Score: 0.3056
   • XGBoost Model F1-Score: 0.5033
   • Net Lift in F1-Score:   +0.1976 (+64.7% improvement)

Detailed Classification Report:
                    precision    recall  f1-score   support

Stable/Growing (0)     0.5390    0.5537    0.5462       912
      Decaying (1)     0.5108    0.4959    0.5033       857

          accuracy                         0.5257      1769
         macro avg     0.5249    0.5248    0.5247      1769
      weighted avg     0.5253    0.5257    0.5254      1769



## 5. Interpretation & Action Engine (Rubric Items 6 & 7)

Interpretation & Action Engine: We extract XGBoost feature importances to see what drives decay predictions. Then, we build a Ranked Action Engine for pages predicted to decay (`xgb_preds == 1`), assigning specific Reason Codes based on feature thresholds, sorted by `prev_clicks` to prioritize high-value traffic at risk.

### Reason Code Heuristics:
1. **`Title/Meta Rewrite (Low CTR)`**: `prev_ctr < 0.02` (High impressions but failing to convert searchers into clicks).
2. **`Content Quality Refresh (Low Engagement)`**: `prev_engagement_rate < 0.10` (Users click through but disengage quickly).
3. **`Competitor Analysis (Drop despite good metrics)`**: Healthy baseline metrics but external rank volatility or competing content.

In [6]:
import matplotlib.pyplot as plt

importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("="*45)
print("     XGBOOST FEATURE IMPORTANCE")
print("="*45)
for feat, imp in importances.items():
    print(f"  • {feat:<22}: {imp:.4f} ({imp*100:.1f}%)")
print("="*45)

df_actions = X_test[xgb_preds == 1].copy()

df_actions['content_hash_id'] = df_model.loc[df_actions.index, 'content_hash_id']
df_actions['client_hash_id'] = df_model.loc[df_actions.index, 'client_hash_id']

def assign_reason_code(row):
    if row['prev_ctr'] < 0.02:
        return 'Title/Meta Rewrite (Low CTR)'
    elif row['prev_engagement_rate'] < 0.10:
        return 'Content Quality Refresh (Low Engagement)'
    else:
        return 'Competitor Analysis (Drop despite good metrics)'

df_actions['reason_code'] = df_actions.apply(assign_reason_code, axis=1)

df_actions = df_actions.sort_values(by='prev_clicks', ascending=False)

print(f"\nTotal Actionable Decaying Pages Identified: {len(df_actions):,}")
print("\nReason Code Breakdown:")
print(df_actions['reason_code'].value_counts())

action_cols = ['content_hash_id', 'prev_clicks', 'prev_ctr', 'prev_engagement_rate', 'reason_code']
print("\n" + "="*80)
print("  TOP 10 RANKED CONTENT REFRESH OPPORTUNITIES (ACTION PLAYBOOK)")
print("="*80)
if 'display' in globals():
    display(df_actions[action_cols].head(10))
else:
    print(df_actions[action_cols].head(10).to_string(index=False))


     XGBOOST FEATURE IMPORTANCE
  • prev_organic_ratio    : 0.1794 (17.9%)
  • prev_avg_position     : 0.1702 (17.0%)
  • prev_clicks           : 0.1662 (16.6%)
  • prev_impressions      : 0.1633 (16.3%)
  • prev_ctr              : 0.1632 (16.3%)
  • prev_engagement_rate  : 0.1577 (15.8%)

Total Actionable Decaying Pages Identified: 832

Reason Code Breakdown:
reason_code
Title/Meta Rewrite (Low CTR)                       795
Content Quality Refresh (Low Engagement)            32
Competitor Analysis (Drop despite good metrics)      5
Name: count, dtype: int64

  TOP 10 RANKED CONTENT REFRESH OPPORTUNITIES (ACTION PLAYBOOK)
         content_hash_id  prev_clicks  prev_ctr  prev_engagement_rate                              reason_code
content_5fa2737c68998c2e        490.0    0.0099                0.0102             Title/Meta Rewrite (Low CTR)
content_b99ea6861864dea5        165.0    0.0020                0.0000             Title/Meta Rewrite (Low CTR)
content_471d9cabce329a66        160.